In [17]:
import os
import chromadb

from groq import Groq

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

# Define the client for groq
client = Groq(api_key=GROQ_API_KEY)

# Define model name
model_name = 'llama-3.3-70b-versatile'
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [18]:
# Making the embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [19]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [20]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hsnw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [21]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [22]:
vectorstore_persisted._collection.count()

3336

In [23]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

query_expansion_user_message = """
<Question>
{question}
</Question>
"""

In [24]:
user_input = "Any specific fines levied on the company in 2022?"

In [25]:
def retrieve_context(user_query, query_expansion_system_message=query_expansion_system_message, query_expansion_user_message=query_expansion_user_message):
    
    prompt = [
        {'role': 'system', 'content': query_expansion_system_message},
        {
            'role': 'user', 'content': query_expansion_user_message.format(
                question=user_query
            )
        }
    ]
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )
    
    response = response.choices[0].message.content.strip().split('\n')
    for i in range(len(response)):
        response[i] = response[i].strip()
        # print(response[i])
    # return response
    context_list = []
    for query in response:
        context_list.extend([d.page_content for d in retriever.invoke(query)])
    return "\n---\n".join(list(set(context_list)))

In [26]:
res = retrieve_context(user_input, query_expansion_system_message, query_expansion_user_message)

In [11]:
res

'a\tsettlement\tfiled\twith\tthe\tCourt\ton\tSeptember\t29,\t2018,\tin\tconnection\twith\tthe\tactions\ttaken\tby\tthe\tSEC\trelating\tto\tMr.\tMusk’s\tstatement\ton\tAugust\t7,\n\t\n2018\tthat\the\twas\tconsidering\ttaking\tTesla\tprivate.\tPursuant\tto\tthe\tsettlement,\twe,\tamong\tother\tthings,\tpaid\ta\tcivil\tpenalty\tof\t$20\tmillion,\tappointed\tan\n\t\nindependent\tdirector\tas\tthe\tchair\tof\tour\tboard\tof\tdirectors,\tappointed\ttwo\tadditional\tindependent\tdirectors\tto\tour\tboard\tof\tdirectors\tand\tmade\tfurther\n\t\nenhancements\tto\tour\tdisclosure\tcontrols\tand\tother\tcorporate\tgovernance-related\tmatters.\tOn\tApril\t26,\t2019,\tthis\tsettlement\twas\tamended\tto\tclarify\n\t\ncertain\tof\tthe\tpreviously-agreed\tdisclosure\tprocedures,\twhich\twas\tsubsequently\tapproved\tby\tthe\tCourt.\tAll\tother\tterms\tof\tthe\tprior\tsettlement\twere\n\t\nreaffirmed\twithout\tmodification.\tAlthough\twe\tintend\tto\tcontinue\tto\tcomply\twith\tthe\tterms\tand\trequirem

In [27]:
type(res)

str

In [28]:
query_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

query_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [29]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{'role': 'user', 'content': 'Hello, are you groq?'}],
    temperature=0.8
)
response.choices[0].message.content

'Hello! I\'m an AI designed to assist and communicate with users in a helpful and informative way. I don\'t have a personal name like "Groq," but I\'m here to help answer your questions and provide information on a wide range of topics. How can I assist you today?'

In [30]:
def respond(user_input, query_system_message=query_system_message, query_user_message_template=query_user_message_template):
    context_for_query = retrieve_context(user_input)
    # print(context_for_query)
    prompt = [
        {'role': 'system', 'content': query_system_message},
        {
            'role': 'user', 'content': query_user_message_template.format(
                context=context_for_query,
                question=user_input
            )
        }
    ]
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=prompt,
            temperature=0
        )

        answer = response.choices[0].message.content.strip()
    except Exception as e:
        answer = f'Sorry, I encountered the following error: \n {e}'
    return answer

In [31]:
respond("What was the total revenue of the company in 2022?")

'$81.46 billion'